<a href="https://colab.research.google.com/github/pranav-1205/CSL701-Roll_No_CS47_Machine-Learning-Lab/blob/main/Experiments/ML_Experiment_7_Principal_Component_Analysis_(PCA).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Machine Learning Experiment Number 7


Student Name : Pranav Vivek Sadwelkar


Batch : CS 03


Roll Number : CS 47




Aim

To implement Principal Component Analysis (PCA) for dimensionality reduction on the Breast Cancer Wisconsin (Diagnostic) dataset, visualize the high-dimensional feature space in a reduced 2D plane, and evaluate its impact on classification performance.

## Objectives

1. To understand the concepts of dimensionality reduction, feature correlation, and Principal Component Analysis.

2. To preprocess, clean, and scale high-dimensional real-world data using StandardScaler.

3. To implement PCA using Python to compress feature dimensions from 30 continuous features down to 2 principal components.

4. To calculate and plot the explained variance ratio and cumulative explained variance.

5. To visualize the transformed 2D dataset and analyze class separability.

6. To evaluate and compare classification model performance (e.g., Logistic Regression) before and after applying PCA.

# Relevant Theory

### Principal Component Analysis (PCA)

Principal Component Analysis (PCA) is an unsupervised machine learning technique used for dimensionality reduction. It transforms a dataset containing a large set of correlated features into a smaller set of uncorrelated variables called Principal Components (PCs), while preserving as much data variance as possible.

Mathematical Steps:

1. Standardization: Scale each feature to have a mean of 0 and a standard deviation of 1.

2. Covariance Matrix: Compute the covariance matrix C to quantify feature correlations:

`C = (1 / (n - 1)) * X^T X`



3. Eigen decomposition: Compute eigenvalues (lambda) and eigenvectors (v) of matrix C:

`C v = lambda * v`



    * Eigenvectors represent the direction of the principal axes.
    * Eigenvalues indicate the amount of variance along those axes.
4. Projection: Project original data X onto top k principal component directions W:

`X_pca = X W`  (matrix product)



###Dataset Information


1. Dataset Name: Breast Cancer Wisconsin (Diagnostic) Dataset

2. Kaggle Link: https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data

3. Classes: 2 binary classes (Malignant [M] and Benign [B])

4. Features: 30 continuous features computed from digitized images of fine needle aspirates (FNA) of a breast mass (e.g., radius_mean, texture_mean, perimeter_mean, area_mean, smoothness_mean, concavity_mean, etc.).

5. Non-Feature Columns: id (identifier) and Unnamed: 32 (empty column).

# Task 1: Import Libraries and Load Dataset

Import core Python libraries (numpy, pandas, matplotlib, seaborn, sklearn) and load data.csv obtained from Kaggle.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# In Google Colab: upload data.csv when prompted. Locally: keep data.csv next to the notebook.
try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    pass

df = pd.read_csv('data.csv')
print('Dataset shape:', df.shape)
display(df.head())

# Task 2: Explore the Dataset

Inspect missing values, view dataset summary statistics, check column data types, and inspect target class distributions (diagnosis).

In [ ]:
print('Missing values:\n', df.isnull().sum())
print('\nData types:\n', df.dtypes)
print('\nSummary statistics:')
display(df.describe())
print('\nDiagnosis distribution:')
counts = df['diagnosis'].value_counts()
print(counts)
plt.figure(figsize=(6,4))
plt.bar(counts.index, counts.values)
plt.xlabel('Diagnosis')
plt.ylabel('Count')
plt.title('Diagnosis Class Distribution')
plt.show()

# Task 3: Data Cleaning and Feature/Target Selection

Drop metadata columns (id and Unnamed: 32). Convert the categorical target column (diagnosis: 'M'/'B') into numerical values (1/0) and separate feature matrix X (30 features) from label vector y.   

In [ ]:
df = df.drop(columns=['id', 'Unnamed: 32'])
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']
print('Feature matrix shape:', X.shape)
print('Target shape:', y.shape)
print('Target values:', y.unique())

# Task 4: Standardize Features

Apply StandardScaler to transform all 30 numeric features so they have zero mean and unit variance (mean = 0, standard deviation = 1).

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Scaled feature matrix shape:', X_scaled.shape)
print('First 5 feature means:', X_scaled.mean(axis=0)[:5])
print('First 5 feature standard deviations:', X_scaled.std(axis=0)[:5])

# Task 5: Implement PCA (30D to 2D)

Use sklearn.decomposition.PCA to reduce feature dimensions from 30 down to 2 principal components (PC1 and PC2).

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
print('Original dimensions:', X_scaled.shape[1])
print('Reduced dimensions:', X_pca.shape[1])
print('First 5 PCA rows:\n', X_pca[:5])

# Task 6: Analyze Explained Variance Ratio

Compute and print the variance explained by PC1 and PC2, along with the cumulative variance retained.


In [ ]:
pc1_var, pc2_var = pca.explained_variance_ratio_
cumulative_var = pca.explained_variance_ratio_.sum()
print(f'PC1 explained variance: {pc1_var:.4f} ({pc1_var*100:.2f}%)')
print(f'PC2 explained variance: {pc2_var:.4f} ({pc2_var*100:.2f}%)')
print(f'Cumulative variance retained: {cumulative_var:.4f} ({cumulative_var*100:.2f}%)')

# Full PCA (all 30 components) to show how variance accumulates
pca_full = PCA().fit(X_scaled)
ratio = pca_full.explained_variance_ratio_
cum = np.cumsum(ratio)
n95 = int(np.argmax(cum >= 0.95)) + 1
print(f'Components needed to retain 95% variance: {n95}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(range(1, len(ratio) + 1), ratio)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Explained Variance per Component')

axes[1].plot(range(1, len(cum) + 1), cum, marker='o')
axes[1].axhline(0.95, color='red', linestyle='--', label='95% threshold')
axes[1].axvline(2, color='gray', linestyle=':', label='2 components (63.24%)')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Task 7: Visualize PCA Transformed Boundary / Scatter Plot

Create a 2D scatter plot with PC1 on the x-axis and PC2 on the y-axis, hue-coded by tumor diagnosis (Malignant vs. Benign).

In [ ]:
pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['Diagnosis'] = y.map({0: 'Benign', 1: 'Malignant'})

plt.figure(figsize=(9,6))
for label, marker in [('Benign', 'o'), ('Malignant', '^')]:
    subset = pca_df[pca_df['Diagnosis'] == label]
    plt.scatter(subset['PC1'], subset['PC2'], marker=marker, s=45, label=label, alpha=0.75)
plt.title('PCA Projection: PC1 vs PC2')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

# Task 8: Train Classifier on Raw High-Dimensional Data (30 Features)

Split the original scaled 30-feature dataset into training and testing sets. Train a Logistic Regression model and obtain baseline performance metrics.

In [ ]:
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

log_raw = LogisticRegression(max_iter=2000, random_state=42)
log_raw.fit(X_train_raw, y_train_raw)
y_pred_raw = log_raw.predict(X_test_raw)

raw_accuracy = accuracy_score(y_test_raw, y_pred_raw)
print(f'Baseline Logistic Regression Accuracy: {raw_accuracy:.4f} ({raw_accuracy*100:.2f}%)')

# Task 9: Train Classifier on PCA Transformed Data (2 Features)

Split the 2-component PCA dataset into training and testing sets. Train a Logistic Regression model using only PC1 and PC2.

In [ ]:
X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)

log_pca = LogisticRegression(max_iter=2000, random_state=42)
log_pca.fit(X_train_pca, y_train_pca)
y_pred_pca = log_pca.predict(X_test_pca)

pca_accuracy = accuracy_score(y_test_pca, y_pred_pca)
print(f'PCA Logistic Regression Accuracy: {pca_accuracy:.4f} ({pca_accuracy*100:.2f}%)')

# Task 10: Compare Classifier Performance

Compare the performance of the model before PCA (30 features) vs. after PCA (2 components) using:   

*   Accuracy Score
*   Confusion Matrix
*   Classification Report (Precision, Recall, F1-Score)  



In [ ]:
print('=== Accuracy Comparison ===')
print(f'Before PCA (30 features): {raw_accuracy:.4f} ({raw_accuracy*100:.2f}%)')
print(f'After PCA (2 components): {pca_accuracy:.4f} ({pca_accuracy*100:.2f}%)')
print(f'Accuracy difference: {(pca_accuracy-raw_accuracy)*100:.2f} percentage points')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, cm, title in [
    (axes[0], confusion_matrix(y_test_raw, y_pred_raw), 'Confusion Matrix - Before PCA'),
    (axes[1], confusion_matrix(y_test_pca, y_pred_pca), 'Confusion Matrix - After PCA')]:
    im = ax.imshow(cm, interpolation='nearest')
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_xticks([0, 1], ['Benign', 'Malignant'])
    ax.set_yticks([0, 1], ['Benign', 'Malignant'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha='center', va='center')
plt.tight_layout()
plt.show()

print('\nClassification Report - Before PCA')
print(classification_report(y_test_raw, y_pred_raw, target_names=['Benign', 'Malignant']))
print('Classification Report - After PCA')
print(classification_report(y_test_pca, y_pred_pca, target_names=['Benign', 'Malignant']))

# Conclusion

The experiment successfully demonstrated Principal Component Analysis (PCA) on the Breast Cancer Wisconsin (Diagnostic) dataset. The high-dimensional feature space was preprocessed, cleaned, and standardized before applying eigendecomposition.

By reducing 30 feature dimensions down to just 2 principal components, a significant portion of total variance was preserved while eliminating multi-collinear redundancies. Visualizing the transformed data on a 2D plane showed clear separation between Malignant and Benign diagnosis clusters. Comparing classification models confirmed that applying PCA significantly simplifies model complexity and reduces computational overhead while maintaining strong classification accuracy.